# ADoRE-DoE Phase 5：ML 殘差修正（Module 2）

`y_final = y_RSM + λ·e_ML`，殘差目標為 out-of-fold。
> 響應由**合成函數（Synthetic）**產生，含一個二次 RSM 無法表示的結構性殘差；需要 `pip install -e ".[plot,ml]"`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
import adore_doe as ad
from adore_doe import design, residual, rsm
SEED = 11

## 1. 資料：45 個 run（I-optimal）＋ 一組獨立驗證資料

In [ ]:
factors = [ad.Factor(c, 0, 10) for c in "ABC"]
quad = lambda Z: 1 + 2*Z[:,0] - Z[:,1] + 0.5*Z[:,2]**2 + 0.8*Z[:,0]*Z[:,1]
truth = lambda Z: quad(Z) + 1.0*np.sin(2.5*Z[:,0])*np.cos(2*Z[:,1])          # 二次 + 結構性殘差
ds = design.i_optimal(factors, "quadratic", runs=45, seed=SEED, n_starts=6)
ds.add_response("y", truth(ds.coded) + 0.05*np.random.default_rng(SEED).standard_normal(45))
V = 2*qmc.Sobol(3, seed=99).random(512) - 1                                  # 獨立驗證點：不參與任何擬合
yv = truth(V)
print(ds.n_runs, "training runs;", len(V), "independent validation points")

## 2. 純 RSM 與混合模型

In [ ]:
base = rsm.fit(ds, "y", model="quadratic")
h = residual.fit(ds, "y", model="quadratic", seed=SEED)
print("RSM: R2 =", round(base.r2, 3), " predicted R2 =", round(base.r2_pred, 3))
print("chosen learner:", h.learner_name, "| lambda =", round(h.lam, 3))
h.report.round(3)

`report`：各學習器在 OOF 殘差上的比較；`gain` 是相對於「不修正」的 RMSE 改善（需 ≥ 5% 才會採用）。

## 3. 獨立驗證資料上的評估

In [ ]:
h.evaluate(V, yv).round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)
for a, (name, p) in zip(ax, [("RSM", h.predict_rsm(V)), ("Hybrid", h.predict(V))]):
    a.scatter(yv, p, s=10); a.plot([yv.min(), yv.max()], [yv.min(), yv.max()], "r"); a.set_title(name + " (validation)")
    a.set_xlabel("Actual"); a.set_ylabel("Predicted")
plt.tight_layout(); plt.savefig("output/residual_validation.png", dpi=150); plt.show()

## 4. 沒有結構時：回退成純 RSM（不會變差）

In [ ]:
ds2 = design.i_optimal(factors, "quadratic", runs=45, seed=SEED, n_starts=6)
ds2.add_response("y", quad(ds2.coded) + 0.05*np.random.default_rng(SEED).standard_normal(45))
h2 = residual.fit(ds2, "y", seed=SEED)
print("chosen learner:", h2.learner_name, "| lambda =", h2.lam)
h2.evaluate(V, quad(V)).round(4)

## 5. 多個響應：每個響應各自決定

In [ ]:
ds.add_response("z", 1e6*(quad(ds.coded)) + 5e4*np.random.default_rng(3).standard_normal(45))
both = residual.fit_responses(ds, ["y", "z"], seed=SEED)
{k: (m.learner_name, round(m.lam, 3)) for k, m in both.items()}

## 6. 提醒

* 殘差目標由折內 RSM 產生，內層的學習器 / λ 比較仍有殘餘樂觀偏差——**請以獨立驗證資料評估**（上面第 3 節）。
* 樣本少時（例如 30 個 run）改善有限，甚至可能變差；完整的 20-seed 實驗結果見 `docs/residual.md`。